# LaunchReady: benchmark dry run on Kaggle T4 x2 (Qwen3-4B)

**Why:** prove `bench/run_session_5090.sh` works end to end on a free GPU before renting an
RTX 5090 (BUILD_PLAN Phase 5 step 5). Same flow as the 5090 session, with Qwen/Qwen3-4B:

1. a startup-only "card" server: Qwen3-4B's card line (expected to **fail** on vLLM 0.32.0, which removed `--enable-reasoning`), then the same line without that flag
2. a vLLM benchmark server that tries FP8 KV first and falls back without it
3. speculative decoding on/off (ngram stands in for NVIDIA's DSpark drafter)
4. `llama-server -np 8` built from llama.cpp v0.8.0, and Ollama at concurrency 1
5. a small GSM8K/IFEval subset through lm-eval, with AIPerf for every benchmark point

Numbers from this notebook are a **dry run of the scripts**, not published results.

**To run:** Kaggle → Settings → Accelerator **GPU T4 x2**, Internet **on** → *Run all*
(~1.5–2.5 h: llama.cpp CUDA build ~15 min, three vLLM startups, Ollama pull).
Then download `/kaggle/working/launchready_dryrun.zip` from the Output panel and share it.

Before running, push the commit named in `LAUNCHREADY_COMMIT` to GitHub.

In [ ]:
import os
import subprocess
import sys

# Pinned versions: change them here and nowhere else.
# Commit with bench/ and the Kaggle matrix (branch phase5-bench); must be pushed to GitHub.
LAUNCHREADY_COMMIT = "8b4ca328e8de8acae2a9837206d9bc836c71e6e0"
PINS = ["torch==2.13.0", "vllm==0.32.0", "uv==0.13.6"]  # vLLM 0.32.0 worked on T4 in Phase 3
REPO_DIR = "/kaggle/working/launchready"

try:
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True, check=True).stdout)
except (FileNotFoundError, subprocess.CalledProcessError) as e:
    raise RuntimeError("No GPU: Settings → Accelerator → GPU T4 x2, then Run all.") from e

if not os.path.isdir(REPO_DIR):
    subprocess.run(
        ["git", "clone", "https://github.com/Tanmaypathak18/launchready", REPO_DIR], check=True
    )
subprocess.run(["git", "-C", REPO_DIR, "fetch", "--all", "-q"], check=True)
subprocess.run(["git", "-C", REPO_DIR, "checkout", "-q", LAUNCHREADY_COMMIT], check=True)
head = subprocess.run(
    ["git", "-C", REPO_DIR, "rev-parse", "HEAD"], capture_output=True, text=True, check=True
).stdout.strip()
assert head == LAUNCHREADY_COMMIT, f"checked out {head}, expected {LAUNCHREADY_COMMIT}"
print(
    subprocess.run(
        ["git", "-C", REPO_DIR, "log", "--oneline", "-1"], capture_output=True, text=True
    ).stdout
)

print("Installing pinned packages (5–10 min)...")
pip = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--progress-bar", "off", *PINS],
    capture_output=True,
    text=True,
)
if pip.returncode:
    print(pip.stdout[-4000:], pip.stderr[-4000:])
    raise RuntimeError("pip install failed (see above). Is Internet on?")
print(subprocess.run(["vllm", "--version"], capture_output=True, text=True).stdout)

## Run the session

This runs exactly what the 5090 session runs, with the Kaggle matrix. `HOURLY_PRICE=0`
(Kaggle is free), so $ per 1M tokens comes out as 0. `MAX_HOURS=3` is the hard stop.
Output streams below; the full logs are under `results_dryrun/bench/<session>/logs/`.

In [ ]:
env = {
    **os.environ,
    "MATRIX": "bench/matrices/kaggle_qwen3_4b.yaml",
    "HOURLY_PRICE": "0",
    "MAX_HOURS": "3",
    "BENCH_PROVIDER": "kaggle",
    "BENCH_CONTAINER_IMAGE": "kaggle notebook (pip vllm==0.32.0)",
}
if "HF_TOKEN" not in env:
    try:
        from kaggle_secrets import UserSecretsClient

        env["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        print("No HF_TOKEN secret (optional: none of these models is gated).")

proc = subprocess.Popen(
    ["bash", "bench/run_session_5090.sh"],
    cwd=REPO_DIR,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
for line in proc.stdout:
    print(line, end="")
print("exit code", proc.wait())

## Pack the results

Zipped first, so there's something to download even if a later cell fails. Download
`/kaggle/working/launchready_dryrun.zip` from the Output panel (not the whole working directory).

In [ ]:
import shutil

os.chdir(REPO_DIR)
if os.path.isdir("results_dryrun"):
    zip_path = shutil.make_archive(
        "/kaggle/working/launchready_dryrun", "zip", REPO_DIR, "results_dryrun"
    )
    print(zip_path, os.path.getsize(zip_path) // 2**20, "MiB")
else:
    print("No results_dryrun/ folder: the session stopped before writing anything (see above).")

## Check the outputs

Prints what exists and reads each file only if it's there, so a partial run still shows its
results. Expected: the `card` and `fp8-kv` variants fail (removed flag; T4 has no FP8), and
everything else passes.

In [ ]:
import glob
import json

import pandas as pd


def show_csv(path, cols=None):
    if not os.path.exists(path):
        print(f"(missing) {path}")
        return None
    df = pd.read_csv(path)
    display(df[[c for c in cols if c in df.columns]] if cols else df)
    return df


sessions = sorted(glob.glob("results_dryrun/bench/dryrun_kaggle_qwen3_4b_*"))
if not sessions:
    print("No session folder under results_dryrun/bench/.")
else:
    session_dir = sessions[-1]
    print("session:", session_dir)
    for root, _, files in sorted(os.walk(session_dir)):
        if "/aiperf/" in root + "/":
            continue
        for f in sorted(files):
            print("  ", os.path.relpath(os.path.join(root, f), session_dir))

    stages = f"{session_dir}/session_stages.tsv"
    print(open(stages).read() if os.path.exists(stages) else f"(missing) {stages}")

    startup = [json.load(open(p)) for p in sorted(glob.glob(f"{session_dir}/startup/*.json"))]
    if startup:
        display(
            pd.DataFrame(
                [
                    {
                        "server": s["server"],
                        "variant": s["variant"],
                        "passed": s["passed"],
                        "health": s["health"],
                        "s_to_healthy": s["seconds_to_healthy"],
                        "engine": s.get("engine_version"),
                        "cmd": " ".join(s["cmd"][:3]),
                    }
                    for s in startup
                ]
            )
        )
    else:
        print("(no startup checks recorded)")
    for p in sorted(glob.glob(f"{session_dir}/errors/*.json")):
        print("ERROR", p, json.load(open(p))["error"])

    # Ollama: containers have no systemd, so the matrix itself must start `ollama serve`.
    print("systemd running:", os.path.isdir("/run/systemd/system"))
    ollama = [s for s in startup if s["server"] == "ollama-gguf"]
    for s in ollama:
        print("ollama started with:", " ".join(s["cmd"]), "| passed:", s["passed"])
    if not ollama:
        print("(Ollama was not started in this session)")
    build_info = "/tmp/llama.cpp/build/LAUNCHREADY_BUILD.txt"
    print(open(build_info).read() if os.path.exists(build_info) else f"(missing) {build_info}")

runs = show_csv(
    "results_dryrun/summary.csv",
    [
        "run_id",
        "status",
        "ttft_ms_p50",
        "ttft_ms_p99",
        "itl_ms_p50",
        "itl_ms_p99",
        "output_tps_total",
        "output_tps_per_user_p50",
        "osl_mean",
        "gpu_mem_used_mib_peak",
        "gpu_power_w_mean",
        "note",
    ],
)
show_csv(
    "results_dryrun/quality_summary.csv",
    ["server", "task", "metric", "value", "thinking", "gguf_tokenizer_pre", "status"],
)
if runs is not None:
    bad = runs[runs["status"] != "ok"]["run_id"].tolist()
    print(f"{len(runs)} runs, {len(bad)} not ok", bad)